# Train DINO + SAM + language-conditioned BC

Run cells individually, in order, on a Colab Enterprise GPU runtime. Do not use Run all for the first session.
DINO, SAM and CLIP are frozen; the BC policy learns actions from RGB, robot state, language and mask features.

Before starting: confirm active Google Cloud credits. Use a GPU runtime such as g2-standard-4 with one NVIDIA L4 where available, with at least 50 GB data disk and idle shutdown enabled. GPU quota and availability depend on the project/region.

Upload `sam_project_source.tar.gz` to the runtime using the notebook's Files pane. Build the archive with `python scripts/sam/package_source.py` to include the current source. Upload it into `/content`.

This notebook prepares offline training and held-out action MSE. The SAM closed-loop evaluator is not included in this branch. Add and validate a separate LIBERO rollout implementation before measuring robot success; see `docs/SAM_GPU_RUNBOOK.md`.


In [ ]:
import os, sys, subprocess, tarfile, hashlib, time
from pathlib import Path

# Adjust only this path if the Files pane uploaded the archive elsewhere.
ARCHIVE = Path("/content/sam_project_source.tar.gz").resolve()
assert ARCHIVE.is_file(), f"Upload the source archive, then set ARCHIVE to its path. Current directory: {Path.cwd()}"
ROOT = ARCHIVE.parent / "open-vocab-robot-learning"
if not ROOT.exists():
    with tarfile.open(ARCHIVE) as archive:
        for member in archive.getmembers():
            destination = (ARCHIVE.parent / member.name).resolve()
            if not destination.is_relative_to(ARCHIVE.parent) or member.issym() or member.islnk():
                raise ValueError("Unexpected archive entry")
        archive.extractall(ARCHIVE.parent, filter="data")
assert (ROOT / "scripts/libero/run_grounded.py").is_file()
ENV = os.environ.copy()
ENV["HF_HOME"] = str(ROOT / "checkpoints/huggingface")
ENV["TORCH_HOME"] = str(ROOT / "checkpoints/torch")
def run(*args):
    return subprocess.run([str(a) for a in args], cwd=ROOT, env=ENV, check=True)
run("nvidia-smi")
print("Project:", ROOT)


## Install an isolated Python environment
This uses Python 3.10 and the team's PyTorch 2.2.2 / torchvision 0.17.2 versions with CUDA 12.1 wheels. The notebook kernel itself is not replaced. All experiment commands use the isolated Python executable.


In [ ]:
run(sys.executable, "-m", "pip", "install", "uv")
if not (ROOT / ".venv-gpu/bin/python").exists():
    run(sys.executable, "-m", "uv", "venv", "--python", "3.10", ".venv-gpu")
PY = ROOT / ".venv-gpu/bin/python"
run(PY, "-c", "import sys; assert sys.version_info[:2] == (3, 10), 'Use a fresh Python 3.10 environment'")
run(sys.executable, "-m", "uv", "pip", "install", "--python", PY,
    "torch==2.2.2", "torchvision==0.17.2", "--index-url", "https://download.pytorch.org/whl/cu121")
run(sys.executable, "-m", "uv", "pip", "install", "--python", PY, "-r", "requirements-sam.txt")
run(sys.executable, "-m", "uv", "pip", "check", "--python", PY)
run(PY, "-c", "import torch, torchvision, numpy, transformers, h5py; from segment_anything import SamPredictor; "
    "assert torch.__version__.split('+')[0] == '2.2.2'; "
    "assert torchvision.__version__.split('+')[0] == '0.17.2'; "
    "assert numpy.__version__ == '1.26.4'; assert transformers.__version__ == '4.46.3'; "
    "assert h5py.__version__ == '3.14.0'; "
    "assert torch.cuda.is_available(), 'GPU not accessible'; "
    "print(torch.__version__, torch.cuda.get_device_name(0))")
run(PY, "-m", "pytest", "tests/test_sam_pipeline.py", "tests/test_grounded_libero.py", "-q")


## Download one task and SAM for the small verification run
The official alphabet-soup task is about 780 MB. DINO, CLIP, and ResNet weights download automatically when first used.


In [ ]:
TASK = "pick_up_the_alphabet_soup_and_place_it_in_the_basket"
run(PY, "scripts/libero/download_object.py", "--tasks", TASK)
checkpoint = ROOT / "checkpoints/sam/sam_vit_b_01ec64.pth"
checkpoint.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint.exists():
    partial = checkpoint.with_suffix(".partial")
    run("curl", "-fL", "--retry", "3", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", "-o", partial)
    partial.replace(checkpoint)
h = hashlib.sha256()
with checkpoint.open("rb") as f:
    for chunk in iter(lambda: f.read(1024*1024), b""):
        h.update(chunk)
assert h.hexdigest() == "ec2df62732614e57411cdcf32a23ffdf28910380d03139ee0f4fcbe91eb8c912"


## Verify perception on six frames
This uses two frames from each of three separate demonstrations. It is a software check, not an experiment result. Inspect the training preview for whether DINO selected the correct target and SAM outlined it. Green = target; blue = basket. SAM confidence is not ground-truth accuracy.


In [ ]:
started = time.monotonic()
run(PY, "scripts/libero/cache_grounding.py", "--splits", "configs/libero_smoke_splits.json",
    "--frame-limit", "2", "--cache-dir", "data/libero/grounding_smoke", "--device", "cuda",
    "--preview-dir", "data/libero/grounding_previews")
print("Smoke cache elapsed seconds (includes initial model downloads):", round(time.monotonic()-started, 1))
from IPython.display import display
from PIL import Image
for path in sorted((ROOT / "data/libero/grounding_previews").glob("*demo_41*.png")):
    display(Image.open(path).resize((768, 384)))


In [ ]:
run(PY, "scripts/libero/run_grounded.py", "--policy", "masks",
    "--splits", "configs/libero_smoke_splits.json", "--frame-limit", "2",
    "--cache-dir", "data/libero/grounding_smoke", "--device", "cuda",
    "--epochs", "1", "--batch-size", "2", "--run-name", "libero_masks_smoke")
run(PY, "scripts/libero/run_grounded.py", "--mode", "evaluate",
    "--splits", "configs/libero_smoke_splits.json", "--frame-limit", "2",
    "--cache-dir", "data/libero/grounding_smoke", "--device", "cuda",
    "--checkpoint", "checkpoints/libero_masks_smoke/best.pt", "--run-name", "libero_masks_smoke")


## Download the full LIBERO-object suite
Proceed after the smoke run passes and the training previews are sensible. All ten official task files total about 7.44 GB. Existing Hugging Face downloads are reused.


In [ ]:
run(PY, "scripts/libero/download_object.py")


## Generate the full perception cache — this can be the longest step
DINO and SAM remain frozen. Reuse this cache for every training epoch and all data budgets. Rerunning with identical settings skips completed frames. Do not change prompts/thresholds midway or add a frame limit here. Inspect additional training/validation previews before committing to a full experiment. Measure actual elapsed time and credit usage; no runtime or cost estimate has been validated on the selected GPU.


In [ ]:
started = time.monotonic()
run(PY, "scripts/libero/cache_grounding.py", "--device", "cuda",
    "--cache-dir", "data/libero/grounding")
print("Full cache elapsed hours:", round((time.monotonic()-started)/3600, 2))


## Train the mask-conditioned BC policy
Start with five epochs, batch size 32. If the GPU runs out of memory, lower the batch size and record that change; keep settings comparable across policies. Use a fresh run name when changing settings. Do not include `--frame-limit` or `--max-batches` in the full training command.


In [ ]:
RUN = "libero_sam_full"
run(PY, "scripts/libero/run_grounded.py", "--policy", "masks", "--epochs", "5",
    "--batch-size", "32", "--device", "cuda", "--run-name", RUN)


## Evaluate held-out action prediction
This reports test MSE, not robot success rate. The selected best validation checkpoint is used.


In [ ]:
run(PY, "scripts/libero/run_grounded.py", "--mode", "evaluate", "--device", "cuda",
    "--checkpoint", f"checkpoints/{RUN}/best.pt", "--run-name", RUN)
import json
print(json.dumps(json.loads((ROOT / f"results/{RUN}/test_results.json").read_text()), indent=2))


## Save experiment deliverables before stopping the runtime
Download the generated archive through the Files pane, or copy it to a Google Cloud Storage bucket. It contains the trained policy, results, selected demonstration IDs, source/configuration files, and dependency versions. It excludes the large datasets and perception cache; preserve those separately on durable storage if you need to resume work. Keep the same source files/paths for resuming the cache because cache provenance is checked.


In [ ]:
freeze = subprocess.check_output([sys.executable, "-m", "uv", "pip", "freeze", "--python", str(PY)], cwd=ROOT, env=ENV, text=True)
(ROOT / f"results/{RUN}/environment.txt").write_text(freeze)
bundle = ROOT.parent / f"{RUN}_deliverables.tar.gz"
with tarfile.open(bundle, "w:gz") as archive:
    for relative in [f"results/{RUN}", f"checkpoints/{RUN}", "configs", "src", "scripts/libero", "requirements-sam.txt"]:
        archive.add(ROOT / relative, arcname=relative)
print("Download this file from the Files pane:", bundle)


## After the first full experiment

- Share the training/validation curves, held-out MSE, and perception previews with the team.
- Add and validate a SAM closed-loop evaluator in a compatible LIBERO simulator environment; it is not included in this branch. Offline MSE does not measure robot success.
- Run the 10%, 25%, 50%, and 100% data budgets only after the full pipeline is validated. `scripts/libero/run_budgets.sh` defaults to the mask policy.
- Stop the GPU runtime explicitly after saving outputs. Idle shutdown is a fallback, not a $50 spending cap. Check storage charges before retaining or deleting resources.

Google documentation: [runtime templates](https://docs.cloud.google.com/colab/docs/create-runtime-template), [idle shutdown](https://docs.cloud.google.com/colab/docs/idle-shutdown).
